In [1]:
#Task 3.1
import pymongo

# connect to MongoDB
client = pymongo.MongoClient("127.0.0.1", 27017)
db = client.get_database("computing")
coll = db.get_collection("problems")

problems = []

# extract data from file
with open("problems.txt", "r") as file:
    content = file.read()

# split into problem blocks, by empty lines
problem_list = content.strip().split("\n\n")

for block in problem_list:
    # extract relevant data from each block
    lines = block.strip().split("\n")  # split the block into 4 lines
    topic = lines[0].replace("Topic: ", "")
    question = lines[1].replace("Question: ", "")
    ans_str = lines[2].replace("Answers: ", "")
    answers = ans_str.split(",")  # split to form a list of strings
    attempts = int(lines[3].replace("Attempts: ", ""))  # no. of attempts as integer
    
    # form a dictionary with the data
    problem = {
        "topic": topic,
        "question": question,
        "answers": answers,
        "attempts": attempts
    }
    problems.append(problem)
    
coll.drop()  # drop collection if exists
result = coll.insert_many(problems)  # insert all problems
print(f"Inserted {len(result.inserted_ids)} problems.")

Inserted 29 problems.


In [2]:
#Task 3.2
query = {
    "topic": {"$in": ["Data Structures", "Algorithms"]},
    "attempts": {"$gte": 5, "$lte": 10}
}

# alternatively
query = {"$and": [{"topic": {"$in": ["Data Structures", "Algorithms"]} },
                  {"attempts": {"$gte": 5, "$lte": 10} }] }

result = coll.find(query)

for item in result:
    print(item["question"])

A memory l___ occurs when dynamically allocated memory is not properly f___. This causes programs to consume increasing amounts of m___.
In a binary search tree, all values in the left subtree are s___ than the root, and all values in the right subtree are l___.
Bubble sort compares a___ elements and s___ them if they are in wrong order.
Merge sort uses d___ and c___ strategy, recursively splitting arrays then m___ sorted subarrays back together.
A stack follows the L___ I___ F___ O___ principle where the last element added is the first one r___.


In [13]:
#Task 3.3
result = coll.delete_many({"topic": "Socket Programming"})

In [4]:
#Task 3.4
import random

# get all problems, storing as list
all_problems = list(coll.find())

# select random problem
selected = random.choice(all_problems)
expected_ans = selected["answers"]

# display problem
print("Topic:", selected["topic"])
print("Question:", selected["question"])
print("\nWhen typing your answers, include the first letter of the word.")

passed = False
while not passed:
    stu_ans = []  # to store student's responses
    # get answers from student
    for i in range(len(expected_ans)):
        ans = input(f"Answer to blank {i + 1}: ")
        stu_ans.append(ans)
    
    # check against expected answers
    correct_count = 0
    for i in range(len(expected_ans)):
        if expected_ans[i].lower() == stu_ans[i].lower():
            correct_count += 1
    
    # calc score
    print(f"You got {correct_count} out of {len(expected_ans)} answers correct.")
    
    selected['attempts'] += 1  # increment number of attempts locally by 1
    
    # update that document in database
    coll.update_one(
        {"_id": selected["_id"]},
        {"$set": {"attempts": selected['attempts']} }  # can also use "$inc": {"attempts": 1} to increment by 1
    )
    
    if correct_count == len(expected_ans):
        print("Excellent!")
        passed = True  # update flag to exit loop
    else:
        print("Let's try again.")
        print()

Topic: Computer Networking
Question: The modern TCP/IP model is an updated version of the original model. It has 5 layers: A___, T___, N___, Data link, and P___ layers. Data is e___ at each layer.

When typing your answers, include the first letter of the word.


Answer to blank 1:  Adam
Answer to blank 2:  Terry
Answer to blank 3:  Natalie
Answer to blank 4:  Perry
Answer to blank 5:  extracted


You got 0 out of 5 answers correct.
Let's try again.



Answer to blank 1:  Application
Answer to blank 2:  Transport
Answer to blank 3:  Network
Answer to blank 4:  Physical
Answer to blank 5:  extracted


You got 4 out of 5 answers correct.
Let's try again.



Answer to blank 1:  Application
Answer to blank 2:  Transport
Answer to blank 3:  Network
Answer to blank 4:  Physical
Answer to blank 5:  encapsulated


You got 5 out of 5 answers correct.
Excellent!


In [5]:
client.close()  # close MongoDB connection